<a href="https://colab.research.google.com/github/Py-saqlain/Movie_Recap/blob/main/MovieApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Connect to your Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Check the GPU
print("\n--- GPU CHECK ---")
!nvidia-smi -L

# 3. Check for your Movie files
import os
folder_path = '/content/drive/MyDrive/MovieApp'
print("\n--- FILE CHECK ---")
try:
    files = os.listdir(folder_path)
    print(f"SUCCESS! The supercomputer sees your folder! Files inside: {files}")
except FileNotFoundError:
    print(f"ERROR: Still can't find {folder_path}.")

Mounted at /content/drive

--- GPU CHECK ---
GPU 0: Tesla T4 (UUID: GPU-99b6f8fa-4be8-d5ab-ca51-0e3d3b7713a0)

--- FILE CHECK ---
SUCCESS! The supercomputer sees your folder! Files inside: ['1920 Evil Returns 2012 Bluray 1080p.srt', '(2012) 1920_ Evil Returns .mp4', 'movie_index.faiss']


In [2]:
# Install zstd dependency for Ollama
!apt-get install zstd -y

# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# 2. Start the engine in the background
print("\nStarting Ollama server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# 3. Download the Llama 3.2 model
print("Downloading Llama 3.2 (Watch how fast this is)...")
!ollama pull llama3.2
print("\n✅ BRAIN SUCCESSFULLY INSTALLED AND RUNNING! DAY 1 IS COMPLETE!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (409 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current us

In [3]:
## DAY_02
!pip install -q sentence-transformers faiss-cpu opencv-python
print("✅ Vision tools installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.5 MB/s eta 0:00:00
✅ Vision tools installed!


In [ ]:
import cv2
import os
# 7,153 seconds * 25 fps = 178,825 total frames

# 1. The EXACT path, including that sneaky space before the .mp4!
video_path = '/content/drive/MyDrive/MovieApp/(2012) 1920_ Evil Returns .mp4'

# 2. HARD CHECK: Does Python actually see this exact file name?
if not os.path.exists(video_path):
    print(f"🛑 ERROR: Still cannot find: {video_path}")
else:
    print("✅ FILE FOUND! The path is perfect. Proceeding...")

    # 3. Create a temporary folder on the supercomputer for the pictures
    frames_dir = '/content/movie_frames'
    os.makedirs(frames_dir, exist_ok=True)

    # 4. Open the movie
    cap = cv2.VideoCapture(video_path)
    fps = round(cap.get(cv2.CAP_PROP_FPS))

    if fps == 0:
        print("🛑 ERROR: I found the file, but OpenCV can't read the video track. Is it fully uploaded?")
    else:
        print(f"Movie opened! Running at {fps} frames per second.")
        print("Starting the slicer. Grabbing 1 frame every second...")

        count = 0
        saved_count = 0

        # 5. The Loop
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break # Movie is over

            # Save 1 frame per second
            if count % fps == 0:
                frame_name = os.path.join(frames_dir, f"frame_{count}.jpg")
                cv2.imwrite(frame_name, frame)
                saved_count += 1

                # Print an update every 500 frames
                if saved_count % 500 == 0:
                    print(f"Sliced {saved_count} frames so far...")

            count += 1

        cap.release()
        print(f"\n✅ DONE! Successfully extracted {saved_count} total frames!")

✅ FILE FOUND! The path is perfect. Proceeding...
Movie opened! Running at 25 frames per second.
Starting the slicer. Grabbing 1 frame every second...
Sliced 500 frames so far...
Sliced 1000 frames so far...
Sliced 1500 frames so far...
Sliced 2000 frames so far...
Sliced 2500 frames so far...
Sliced 3000 frames so far...
Sliced 3500 frames so far...
Sliced 4000 frames so far...
Sliced 4500 frames so far...
Sliced 5000 frames so far...
Sliced 5500 frames so far...
Sliced 6000 frames so far...
Sliced 6500 frames so far...
Sliced 7000 frames so far...

✅ DONE! Successfully extracted 7153 total frames!


In [ ]:
import os
import faiss
from sentence_transformers import SentenceTransformer
from PIL import Image
import re

print("1. Waking up the Vision AI (CLIP)...")
# This downloads the AI that turns images into math
model = SentenceTransformer('clip-ViT-B-32')

frames_dir = '/content/movie_frames'
db_path = '/content/drive/MyDrive/MovieApp/movie_index.faiss'

# This helper function makes sure frame_2 comes before frame_10
def alphanum_key(s):
    return [int(c) if c.isdigit() else c for c in re.split('([0-9]+)', s)]

image_files = sorted([f for f in os.listdir(frames_dir) if f.endswith('.jpg')], key=alphanum_key)
print(f"2. Found {len(image_files)} pictures! Starting the AI visual scan...")

# Setup the math vault (CLIP uses 512 dimensions)
dimension = 512
index = faiss.IndexFlatIP(dimension)

# We will process them in chunks of 128 so we don't blow up the GPU memory
batch_size = 128

for i in range(0, len(image_files), batch_size):
    batch_files = image_files[i:i+batch_size]

    # Open the batch of images
    images = [Image.open(os.path.join(frames_dir, img)) for img in batch_files]

    # AI MAGIC: Look at the images and turn them into math!
    batch_embeddings = model.encode(images)

    # Save the math into the vault
    faiss.normalize_L2(batch_embeddings)
    index.add(batch_embeddings)

    print(f"👀 AI has scanned and memorized {min(i+batch_size, len(image_files))} / {len(image_files)} frames...")

# 3. Save the vault permanently to your Google Drive!
faiss.write_index(index, db_path)
print(f"\n✅ DAY 2 COMPLETE! Database successfully saved to your Google Drive at: {db_path}")

1. Waking up the Vision AI (CLIP)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


2. Found 7153 pictures! Starting the AI visual scan...
👀 AI has scanned and memorized 128 / 7153 frames...
👀 AI has scanned and memorized 256 / 7153 frames...
👀 AI has scanned and memorized 384 / 7153 frames...
👀 AI has scanned and memorized 512 / 7153 frames...
👀 AI has scanned and memorized 640 / 7153 frames...
👀 AI has scanned and memorized 768 / 7153 frames...
👀 AI has scanned and memorized 896 / 7153 frames...
👀 AI has scanned and memorized 1024 / 7153 frames...
👀 AI has scanned and memorized 1152 / 7153 frames...
👀 AI has scanned and memorized 1280 / 7153 frames...
👀 AI has scanned and memorized 1408 / 7153 frames...
👀 AI has scanned and memorized 1536 / 7153 frames...
👀 AI has scanned and memorized 1664 / 7153 frames...
👀 AI has scanned and memorized 1792 / 7153 frames...
👀 AI has scanned and memorized 1920 / 7153 frames...
👀 AI has scanned and memorized 2048 / 7153 frames...
👀 AI has scanned and memorized 2176 / 7153 frames...
👀 AI has scanned and memorized 2304 / 7153 frames..

In [ ]:
# day 3

In [5]:
import faiss
import os
from sentence_transformers import SentenceTransformer

# 1. Re-installing the brain-to-vision tools
!pip install -q sentence-transformers faiss-cpu

# 2. Load the CLIP model (The Translator)
print("Loading CLIP Vision model...")
search_model = SentenceTransformer('clip-ViT-B-32')

# 3. Load the Vault from your Google Drive
db_path = '/content/drive/MyDrive/MovieApp/movie_index.faiss'

if os.path.exists(db_path):
    index = faiss.read_index(db_path)
    print("✅ VISUAL DATABASE LOADED! The AI can now 'see' the movie again.")
else:
    print("🛑 ERROR: I can't find 'movie_index.faiss' on your Drive. Did you move it?")

Loading CLIP Vision model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ VISUAL DATABASE LOADED! The AI can now 'see' the movie again.


In [7]:
import os
import re

srt_path = '/content/drive/MyDrive/MovieApp/1920 Evil Returns 2012 Bluray 1080p.srt'

def scorched_earth_cleaner(path):
    if not os.path.exists(path):
        return None

    # Read the file as raw bytes to avoid encoding errors
    with open(path, 'rb') as f:
        raw_data = f.read().decode('utf-8', errors='ignore')

    # 1. Kill the timestamps (00:00:00,000 --> 00:00:00,000)
    text = re.sub(r'\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}', '', raw_data)

    # 2. Kill the subtitle index numbers
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)

    # 3. Keep ONLY letters, numbers, and basic punctuation
    # This removes the weird "::," and "-->" leftovers
    text = re.sub(r'[^a-zA-Z0-9\s\?\.!,\'\"]', '', text)

    # 4. Collapse multiple spaces into one
    clean_text = " ".join(text.split())

    return clean_text

# Execute
movie_story = scorched_earth_cleaner(srt_path)

if movie_story:
    print("✅ PROBLEM SOLVED. THE TEXT IS RAW AND CLEAN.")
    print("-" * 30)
    print(f"Sample: {movie_story[:300]}")
else:
    print(f"🛑 Still can't find that file at {srt_path}")

✅ PROBLEM SOLVED. THE TEXT IS RAW AND CLEAN.
------------------------------
Sample: 1 000219,250 000223,457 Bhola, is the news good or bad? 2 000224,083 000227,332 Ma'am, the news is good as well as bad. 3 000228,250 000229,915 Stop talking in riddles. 4 000231,625 000234,874 The good news is that Mr. Jaidev is alive. 5 000236,666 000240,040 And the bad news is that he is leading a


In [8]:
import requests
import json
import os

# 1. Ask the user for the genre
print("🎬 GENRE & BGM SELECTION")
print("Available genres: romantic, sad, horror, action, thriller, scifi")
selected_genre = input("Enter the genre for this movie: ").lower().strip()

# 2. Lock in the music path for Day 4 Audio Sync
music_path = f'/content/drive/MyDrive/MovieApp/Music/{selected_genre}.mp3'
print(f"🎵 BGM Track assigned: {music_path}")
print(f"(Reminder: Ensure {selected_genre}.mp3 is uploaded to your Drive's Music folder!)")
print("-" * 30)

def generate_dynamic_script(text, genre):
    # Notice how we inject {genre} into the prompt so Llama shifts its tone!
    prompt = f"""
    [SYSTEM: YOU ARE A PROFESSIONAL YOUTUBE MOVIE RECAPPER. OUTPUT ONLY THE SCRIPT. IT MUST LOOK LIKE A PROFESSIONAL STORYTELLING.]

    TASK: Write a 15-minute high-quality {genre} movie recap.
    DATA: {text[:8000]}

    STRICT RULES:
    1. Match the tone perfectly to a {genre} film.
    2. Ignore all numbers like '000219' or '1, 2, 3'. They are data errors.
    3. Start the script immediately with a hook: "The story begins with..."
    4. Do NOT mention you are an AI.
    5. Do NOT say "Here is your script."
    6. Write in an engaging, captivating storytelling tone.
    """

    payload = {
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False
    }

    print(f"🧠 Llama is reading the story and writing a {genre.upper()} recap... (1-2 mins)")
    response = requests.post("http://localhost:11434/api/generate", json=payload)
    return response.json()['response']

# 3. Run the script generator
final_script = generate_dynamic_script(movie_story, selected_genre)

print(f"\n--- YOUR PROFESSIONAL {selected_genre.upper()} RECAP SCRIPT ---")
print(final_script)

🎬 GENRE & BGM SELECTION
Available genres: romantic, sad, horror, action, thriller, scifi
Enter the genre for this movie: horror
🎵 BGM Track assigned: /content/drive/MyDrive/MovieApp/Music/horror.mp3
(Reminder: Ensure horror.mp3 is uploaded to your Drive's Music folder!)
------------------------------
🧠 Llama is reading the story and writing a HORROR recap... (1-2 mins)

--- YOUR PROFESSIONAL HORROR RECAP SCRIPT ---
The story begins with a haunting revelation: a woman's life is a twisted tapestry of love, loss, and longing. Her name is Karuna, and her existence is a maze of dark corridors and hidden truths.

She stands at the threshold of a new chapter, her heart heavy with the weight of memories. The ghosts of her past linger, refusing to be silenced. Mr. Jaidev, a man she's never met, holds a piece of her soul. His presence is a bittersweet reminder of what she's lost.

Karuna's thoughts are a jumble of emotions: guilt, sorrow, and a deep-seated longing for connection. Her mind is a b

In [10]:
# cell 09
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
import re
import os

print("🔍 Waking up the RAG Search Engine (CLIP)...")
# Load the Vision Translator
search_model = SentenceTransformer('clip-ViT-B-32')

# Load your visual vault
db_path = '/content/drive/MyDrive/MovieApp/movie_index.faiss'
if not os.path.exists(db_path):
    print(f"🛑 ERROR: Cannot find {db_path}. Did you mount your drive?")
else:
    index = faiss.read_index(db_path)

    # 1. Split Llama's script into individual sentences
    # This regex splits by periods, exclamation marks, or question marks
    sentences = [s.strip() for s in re.split(r'(?<=[.!?]) +', final_script) if len(s.strip()) > 10]

    print(f"✅ Script split into {len(sentences)} distinct scenes. Starting the visual hunt...\n")

    # 2. The Timeline Vault
    # This list will hold the exact blueprint for tomorrow's audio/video render
    recap_timeline = []

    for sentence in sentences:
        # A. Turn the text into math
        text_embedding = search_model.encode([sentence])
        faiss.normalize_L2(text_embedding)

        # B. Search the FAISS vault for the #1 closest matching frame
        distances, indices = index.search(text_embedding, k=1)

        # C. The matching index IS the timestamp in seconds!
        best_second = int(indices[0][0])

        # D. Save it to our timeline
        recap_timeline.append({
            "text": sentence,
            "timestamp": best_second
        })

    print("🎉 VIDEO RAG COMPLETE! Here is a preview of your sync timeline:\n")
    print("-" * 60)

    # Let's print the first 5 mappings so you can see it working
    for item in recap_timeline[:5]:
        mins = item['timestamp'] // 60
        secs = item['timestamp'] % 60
        print(f"⏱️ Video jumps to [{mins:02d}:{secs:02d}] -> AI Voice says: \"{item['text']}\"")

    print("-" * 60)
    print(f"(...and {len(recap_timeline) - 5} more lines successfully mapped to timestamps!)")


🔍 Waking up the RAG Search Engine (CLIP)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Script split into 17 distinct scenes. Starting the visual hunt...

🎉 VIDEO RAG COMPLETE! Here is a preview of your sync timeline:

------------------------------------------------------------
⏱️ Video jumps to [43:07] -> AI Voice says: "The story begins with a haunting revelation: a woman's life is a twisted tapestry of love, loss, and longing."
⏱️ Video jumps to [08:19] -> AI Voice says: "Her name is Karuna, and her existence is a maze of dark corridors and hidden truths.

She stands at the threshold of a new chapter, her heart heavy with the weight of memories."
⏱️ Video jumps to [54:20] -> AI Voice says: "The ghosts of her past linger, refusing to be silenced."
⏱️ Video jumps to [37:19] -> AI Voice says: "Jaidev, a man she's never met, holds a piece of her soul."
⏱️ Video jumps to [91:49] -> AI Voice says: "His presence is a bittersweet reminder of what she's lost.

Karuna's thoughts are a jumble of emotions: guilt, sorrow, and a deep-seated longing for connection."
--------------

In [11]:
import json

# Define the save path in your Drive
timeline_path = '/content/drive/MyDrive/MovieApp/recap_timeline.json'

# Save the timeline dictionary as a JSON file
with open(timeline_path, 'w') as f:
    json.dump(recap_timeline, f, indent=4)

print(f"✅ TIMELINE BLUEPRINT SAVED SECURELY TO DRIVE!")
print(f"Location: {timeline_path}")
print("-" * 30)
print("DAY 3 cell 10 IS OFFICIALLY COMPLETE. 🏆")

✅ TIMELINE BLUEPRINT SAVED SECURELY TO DRIVE!
Location: /content/drive/MyDrive/MovieApp/recap_timeline.json
------------------------------
DAY 3 cell 10 IS OFFICIALLY COMPLETE. 🏆
